# Week 4 — Embeddings, Similarity & Modalities
### Subtitle: How AI represents meaning in vector space

<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/lectures/week4_embeddings_similarity.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

### Learning Objectives
- Explain what embeddings are and why AI models use them.
- Describe cosine similarity and neighborhoods in embedding space.
- Visualize how meaning is represented geometrically.
- Compare semantic vs. keyword similarity in search.
- Recognize multimodal embeddings as the same idea across modalities.
- Connect embeddings to the Unifying System Diagram.

In [ ]:
# @title Setup (Run this first)
!git clone -q https://github.com/tulane-intro-ai-engineering/main.git
import sys; sys.path.append('/content/main')
from course_utils import lab4_setup, show_mermaid
lab4_setup()
print('✅ Environment ready!')

## 💡 Motivating Example — Mapping Meaning

When you ask an AI:
- “What’s the capital of France?” → it confidently says *Paris*.
- “What’s the capital of bananas?” → it gets confused.

Yet both are just words!

**Guiding Question:**

What if we could place every word, image, or sound in a giant map — where 'closeness' means 'similar meaning'? This is the idea behind **embeddings** — vector representations that encode semantic relationships.

In [ ]:
# @title 🌍 Motivating Visualization: A Tiny Map of Meaning
import matplotlib.pyplot as plt
import numpy as np

words = ['Paris', 'France', 'banana', 'desk', 'Eiffel Tower']
vectors = np.array([
    [0.9, 0.8], [0.85, 0.75], [-0.4, 0.9], [-0.8, 0.7], [0.95, 0.85]
])

plt.figure(figsize=(6,5))
plt.scatter(vectors[:,0], vectors[:,1], c='purple')
for i, w in enumerate(words):
    plt.text(vectors[i,0]+0.02, vectors[i,1], w)
plt.title('A Tiny Map of Meaning (Toy Embedding Space)')
plt.xlabel('Dimension 1'); plt.ylabel('Dimension 2')
plt.show()

In [ ]:
# @title 🔍 Demo: Cosine Similarity Between Words
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

doctor = np.array([0.8, 0.6])
nurse = np.array([0.75, 0.65])
carrot = np.array([-0.2, 0.9])

for word, vec in {'nurse': nurse, 'carrot': carrot}.items():
    sim = cosine_similarity(doctor, vec)
    print(f'Similarity(doctor, {word}) = {sim:.3f}')

In [ ]:
# @title 🧮 Visualizing Embedding Neighborhoods
import matplotlib.pyplot as plt
words = ['doctor', 'nurse', 'hospital', 'car', 'banana']
embeddings = np.array([
    [0.8, 0.6], [0.75, 0.65], [0.7, 0.7], [-0.2, 0.9], [-0.5, 0.8]
])
plt.scatter(embeddings[:,0], embeddings[:,1], c='blue')
for i, w in enumerate(words):
    plt.text(embeddings[i,0]+0.02, embeddings[i,1], w)
plt.xlabel('Dimension 1'); plt.ylabel('Dimension 2')
plt.title('Simple 2D Embedding Space')
plt.show()

In [ ]:
# @title 🧪 Demo: OpenAI Embeddings Example
from openai import OpenAI
client = OpenAI()

texts = ['Tulane University', 'New Orleans', 'banana']
embeds = [client.embeddings.create(input=t, model='text-embedding-3-small').data[0].embedding for t in texts]

def similarity(i, j):
    return cosine_similarity(np.array(embeds[i]), np.array(embeds[j]))

for i in range(len(texts)):
    for j in range(i+1, len(texts)):
        print(f'Similarity({texts[i]}, {texts[j]}) = {similarity(i,j):.3f}')

In [ ]:
# @title 🌐 Optional Visualization: PCA Projection of OpenAI Embeddings
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
proj = pca.fit_transform(np.array(embeds))
plt.figure(figsize=(5,4))
plt.scatter(proj[:,0], proj[:,1], color='green')
for i, txt in enumerate(texts):
    plt.text(proj[i,0]+0.01, proj[i,1], txt)
plt.title('PCA Projection of Embeddings')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.show()

In [ ]:
# @title 🧠 Demo: Keyword vs Semantic Search
from course_utils import simple_keyword_search, semantic_search

query = 'weather in uptown New Orleans'
docs = [
    'Tulane campus is beautiful on sunny days.',
    'The weather in New Orleans is unpredictable.',
    'Rainy season affects downtown traffic.'
]

print('🔍 Keyword Search Results:')
print(simple_keyword_search(query, docs))

print('\n🔍 Semantic Search Results:')
print(semantic_search(query, docs))

In [ ]:
# @title 🧠 Multimodal Embeddings Example (Conceptual)
print('Imagine CLIP embeddings placing both text and image vectors in one space.')
print('A picture of a dog and the word “dog” would have nearby vectors.')

In [ ]:
# @title 🧩 Unifying Diagram v3 — Embedding Store Added
show_mermaid('''graph TD
    subgraph User Interaction
    U[👤 User Query]:::user --> IH[Input Handling]:::process
    end
    subgraph Prompt & Control
    IH --> PC[Prompt & Control]:::control
    end
    subgraph Embedding Store
    PC --> VE[Vector Embeddings]:::embedding
    VE --> VS[Vector Search (Top-k)]:::embedding
    end
    subgraph Core LLM
    VS --> LLM[LLM]:::model
    end
    subgraph Output & Monitoring
    LLM --> OP[Output Processing]:::output
    end
    classDef user fill:#d1e7dd,stroke:#333,stroke-width:1px;
    classDef process fill:#e2e3e5,stroke:#333,stroke-width:1px;
    classDef control fill:#cfe2ff,stroke:#333,stroke-width:1px;
    classDef embedding fill:#fff3cd,stroke:#333,stroke-width:1px;
    classDef model fill:#f8d7da,stroke:#333,stroke-width:1px;
    classDef output fill:#e9ecef,stroke:#333,stroke-width:1px;
''')